# 2 · Agentes de IA — Ciclo ReAct manual
### Sesión 1 — Agentes de IA, Orquestación y Protocolos

**Complejidad: 🟢 Básica**   |   **Dependencias: solo `anthropic`**

Construimos, sin ningún framework, el loop **razonar → actuar → observar** que vimos en el diagrama de la charla. La diferencia con el notebook 1 es que aquí el ciclo se repite **N veces** hasta que el modelo decide que ya tiene todo lo necesario.

Este notebook es completamente autocontenido (no depende de haber corrido el notebook 1).


## 🎯 Objetivo de aprendizaje

Al terminar este notebook vas a poder:
- Explicar la diferencia entre un chatbot y un **agente de IA**.
- Describir el ciclo ReAct (razonar → actuar → observar) y por qué es la base de cualquier agente.
- Implementar ese ciclo a mano, sin frameworks, controlando explícitamente cuándo se detiene (respuesta final vs. límite de iteraciones).


## 📚 Teoría: Agentes de IA y el ciclo ReAct

Un **agente de IA** no es solo un modelo que responde una pregunta — es un modelo que puede **decidir una secuencia de acciones** para lograr un objetivo, usando herramientas y observando los resultados de sus propias acciones antes de decidir el siguiente paso.

El patrón que sostiene esto se llama **ReAct** (*Reason + Act*): el modelo alterna entre razonar ("¿qué necesito hacer ahora?") y actuar (invocar una herramienta), usando lo que observa en cada resultado para decidir si ya puede responder o si necesita otro paso. Es el mismo ciclo de function calling del notebook 1, pero repetido **N veces** en vez de una sola vuelta.

El ciclo termina en alguno de estos tres casos:
- El modelo decide que ya tiene toda la información necesaria y da la respuesta final.
- Se alcanza un límite explícito de iteraciones (`max_steps`) — un control de seguridad imprescindible para evitar loops infinitos.
- Ocurre un error que requiere intervención humana.

**Niveles de autonomía:** un agente puede diseñarse en un espectro que va desde *asistido* (una persona ejecuta cada acción manualmente, el modelo solo sugiere) hasta *autónomo* (decide y ejecuta sin supervisión, con monitoreo posterior). A mayor autonomía, mayor productividad — pero también mayor riesgo, por lo que la elección del nivel correcto depende del caso de uso.


## 0. Instalación (única dependencia)

In [ ]:
!pip install -q anthropic

### Configurar API key de Anthropic

**Cómo obtenerla:** [console.anthropic.com](https://console.anthropic.com/settings/keys)

Recomendado en Colab: guárdala en **Secrets** (ícono de llave 🔑 a la izquierda) con el nombre `ANTHROPIC_API_KEY` y actívala para este notebook. Si no usas Secrets, te la pedirá por input.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = os.environ.get("ANTHROPIC_API_KEY") or getpass("Pega tu ANTHROPIC_API_KEY: ")

print("API key configurada:", "OK" if os.environ.get("ANTHROPIC_API_KEY") else "FALTA")


## 1. Herramientas del agente

In [ ]:
import anthropic

client = anthropic.Anthropic()

def calculadora(expresion: str) -> str:
    """Evalúa una expresión matemática simple."""
    try:
        return str(eval(expresion, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"

def buscar_evento_universidad(tema: str) -> str:
    """Simula una búsqueda en el calendario de eventos de la universidad."""
    eventos = {
        "ia": "Seminario de Inteligencia Artificial - 25 de julio, Auditorio Principal, 3pm",
        "emprendimiento": "Feria de Emprendimiento - 30 de julio, Plazoleta Central, 9am",
    }
    for k, v in eventos.items():
        if k in tema.lower():
            return v
    return "No se encontraron eventos relacionados con ese tema."

agent_tools = [
    {
        "name": "calculadora",
        "description": "Evalúa expresiones matemáticas. Úsala para cualquier cálculo numérico.",
        "input_schema": {"type": "object", "properties": {"expresion": {"type": "string"}}, "required": ["expresion"]}
    },
    {
        "name": "buscar_evento_universidad",
        "description": "Busca eventos en el calendario de la universidad por tema.",
        "input_schema": {"type": "object", "properties": {"tema": {"type": "string"}}, "required": ["tema"]}
    },
]

tool_implementations = {
    "calculadora": lambda expresion: calculadora(expresion),
    "buscar_evento_universidad": lambda tema: buscar_evento_universidad(tema),
}


## 2. El loop del agente

In [ ]:
def run_agent(user_message: str, tools, implementations, max_steps: int = 5, verbose: bool = True):
    """Implementación manual del ciclo ReAct: razonar -> actuar -> observar."""
    messages = [{"role": "user", "content": user_message}]

    for step in range(1, max_steps + 1):
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )

        # -- RAZONAR: si el modelo ya no necesita herramientas, terminamos --
        if response.stop_reason != "tool_use":
            texto_final = "".join(b.text for b in response.content if b.type == "text")
            if verbose:
                print(f"[Paso {step}] Respuesta final del agente.")
            return texto_final

        messages.append({"role": "assistant", "content": response.content})

        # -- ACTUAR + OBSERVAR: ejecutamos cada tool_use solicitado --
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                if verbose:
                    print(f"[Paso {step}] Actuando: {block.name}({block.input})")
                fn = implementations.get(block.name)
                result = fn(**block.input) if fn else f"Herramienta desconocida: {block.name}"
                if verbose:
                    print(f"[Paso {step}] Observando resultado: {result}")
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})

        messages.append({"role": "user", "content": tool_results})

    return "Se alcanzó el límite de pasos (max_steps) sin una respuesta final."


respuesta = run_agent(
    "Busca si hay algún evento de IA en la universidad, y si lo hay, dime cuántos días faltan si hoy es 18 de julio.",
    agent_tools, tool_implementations
)
print("\n=== RESPUESTA FINAL ===")
print(respuesta)


## 🧪 Ejercicio

Modifica `run_agent` para que imprima un **contador de tokens acumulado** en cada paso (usa `response.usage.input_tokens` y `response.usage.output_tokens`). Esto es exactamente lo que herramientas de observabilidad como LangSmith o Langfuse hacen automáticamente — lo veremos en la Sesión 2.

---
**Guarda este notebook a mano** — la función `run_agent` la vamos a reutilizar tal cual en el notebook 6 (Taller 5).

**Siguiente notebook:** `langchain_agente.ipynb` — el mismo agente, reconstruido con un framework de orquestación.


In [ ]:
# Tu código aquí
